# Dataset PSNR Evaluation

Load a training experiment config and a Lightning checkpoint, denoise every particle in the dataset, and compute **PSNR** against clean references.

## Why it was slow

Each particle has **40 frame splits**. The training config uses `split_batch_size=6`, so evaluation was running **7 SCUNet forward passes per particle** × **40,000 particles** (~45 h at ~4 s/particle).

The notebook now sets `INFERENCE_SPLIT_BATCH_SIZE = "all"` so all 40 splits run in **one forward pass** (~5–7× faster). Mixed precision (`USE_AMP`) gives another ~1.5–2× on GPU.

**Configure** paths and speed knobs in the next cell, then re-run from the model-loading cell onward.

In [ ]:
import os
from pathlib import Path

# --- user settings ---------------------------------------------------------
CONFIG_PATH = Path(
    "../configs_train/experiment/10948_sim/multinoise2noise_ctf/scunet.yaml"
)
EXPERIMENT_OVERRIDE = "10948_sim/multinoise2noise_ctf/scunet"

CHECKPOINT_PATH = (
    Path(os.environ["CRYO_DATASETS_DIR"]) / "checkpoints/10948_sim/scunet_newest.ckpt"
)

PARTICLES_STAR_PATH = None
OUTPUT_CSV = Path("../results/10948_sim_scunet_psnr.csv")
CHECKPOINT_RESULTS_CSV = Path("../results/10948_sim_scunet_psnr.partial.csv")

USE_ALL_PARTICLES = True
MAX_PARTICLES = None
BATCH_SIZE = 4
NUM_WORKERS = 8
DEVICE = "cuda"

# --- speed knobs -----------------------------------------------------------
# Biggest win: run all frame splits in one SCUNet forward pass at eval time.
# Training uses split_batch_size=6 for memory; inference can use all splits.
INFERENCE_SPLIT_BATCH_SIZE = "all"  # "all", or an int like 20 / 40

USE_AMP = True              # mixed precision on CUDA
USE_TORCH_COMPILE = False   # set True on PyTorch 2+ if first-batch compile cost is OK
SAVE_EVERY_N = 500          # flush partial CSV every N particles (0 = disable)
# ---------------------------------------------------------------------------

assert CONFIG_PATH is not None or EXPERIMENT_OVERRIDE

In [ ]:
from __future__ import annotations

import math
import os
import sys
from typing import Any

import hydra
import pandas as pd
import torch
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import DictConfig, OmegaConf
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("PROJECT_ROOT", str(PROJECT_ROOT))

from source._noise2noise.multinoise2noise import normalize_for_metrics


def experiment_override_from_config_path(config_path: Path) -> str:
    exp_root = (PROJECT_ROOT / "configs_train" / "experiment").resolve()
    rel = config_path.resolve().relative_to(exp_root)
    return str(rel.with_suffix("")).replace("\\", "/")


def load_train_config(config_path: Path | None, experiment_override: str | None) -> DictConfig:
    if not OmegaConf.has_resolver("sanitize_filename"):
        OmegaConf.register_new_resolver(
            "sanitize_filename", lambda x: str(x).replace("/", "_")
        )

    if GlobalHydra.instance().is_initialized():
        GlobalHydra.instance().clear()

    overrides: list[str] = []
    if config_path is not None:
        overrides.append(f"experiment={experiment_override_from_config_path(config_path)}")
    elif experiment_override:
        overrides.append(f"experiment={experiment_override}")

    config_dir = PROJECT_ROOT / "configs_train"
    with initialize_config_dir(config_dir=str(config_dir.resolve()), version_base="1.3"):
        cfg = compose(config_name="train.yaml", overrides=overrides)

    if PARTICLES_STAR_PATH is not None:
        cfg.datamodule.particles_star_path = str(PARTICLES_STAR_PATH)

    return cfg

In [ ]:
cfg = load_train_config(CONFIG_PATH, EXPERIMENT_OVERRIDE)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}\n"
        "Update CHECKPOINT_PATH in the settings cell."
    )

device = torch.device(DEVICE if torch.cuda.is_available() or DEVICE == "cpu" else "cpu")
experiment_name = (
    experiment_override_from_config_path(CONFIG_PATH)
    if CONFIG_PATH is not None
    else EXPERIMENT_OVERRIDE
)
print("Resolved experiment:", experiment_name)
print("Dataset path:", cfg.datamodule.dataset_path)
print("Star file:", cfg.datamodule.particles_star_path)
print("Lightning model:", cfg.lightning_model._target_)
print("Checkpoint:", CHECKPOINT_PATH)
print("Device:", device)

In [ ]:
datamodule = hydra.utils.instantiate(cfg.datamodule)
datamodule.setup()

particles = datamodule.particles
if "rlnCleanImage" not in particles.columns and "clean_path" not in particles.columns:
    raise ValueError("Star file has no clean references (rlnCleanImage / clean_path).")

if USE_ALL_PARTICLES:
    eval_particles = particles.copy()
else:
    eval_half = int(cfg.lightning_model.eval_half)
    eval_particles = particles[particles["rlnRandomSubset"] == eval_half].copy()

if MAX_PARTICLES is not None:
    eval_particles = eval_particles.iloc[: int(MAX_PARTICLES)].copy()

print(f"Evaluating {len(eval_particles)} / {len(particles)} particles")

eval_dataset = hydra.utils.call(
    cfg.datamodule.datasets.val,
    data=eval_particles,
    dose_weights=datamodule.dose_weights,
    _recursive_=True,
)

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "shuffle": False,
    "pin_memory": device.type == "cuda",
}
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = 2

eval_loader = DataLoader(eval_dataset, **loader_kwargs)

In [ ]:
model_cls = hydra.utils.get_class(cfg.lightning_model._target_)
model = model_cls.load_from_checkpoint(str(CHECKPOINT_PATH), map_location=device)
model = model.to(device)
model.eval()

# Inference-only speed settings (do not affect metric definition)
train_split_batch_size = int(getattr(model.hparams, "split_batch_size", 1))
n_splits = int(cfg.datamodule.splits)
if INFERENCE_SPLIT_BATCH_SIZE == "all":
    inference_split_batch_size = n_splits
else:
    inference_split_batch_size = int(INFERENCE_SPLIT_BATCH_SIZE)

model.hparams.split_batch_size = inference_split_batch_size
model.hparams.use_gradient_checkpointing = False

if USE_TORCH_COMPILE and hasattr(torch, "compile"):
    model.model = torch.compile(model.model, mode="reduce-overhead")

n_forward_passes = (n_splits + inference_split_batch_size - 1) // inference_split_batch_size
print("Loaded checkpoint epoch:", getattr(model, "current_epoch", "unknown"))
print("Reconstruction mode:", model.hparams.reconstruction_mode)
print(f"Training split_batch_size: {train_split_batch_size} -> inference: {inference_split_batch_size}")
print(f"SCUNet forward passes per particle: {n_forward_passes} (was ~{(n_splits + train_split_batch_size - 1) // train_split_batch_size} during training-style eval)")
print("AMP:", USE_AMP and device.type == "cuda")
print("torch.compile:", USE_TORCH_COMPILE)

In [ ]:
autocast_ctx = (
    torch.autocast(device_type=device.type, dtype=torch.float16)
    if USE_AMP and device.type == "cuda"
    else torch.autocast(device_type="cpu", enabled=False)
)


def _move_batch_to_device(batch: dict[str, Any], device: torch.device) -> dict[str, Any]:
    out: dict[str, Any] = {}
    for key, value in batch.items():
        if torch.is_tensor(value):
            out[key] = value.to(device, non_blocking=device.type == "cuda")
        elif isinstance(value, dict):
            out[key] = {
                k: (v.to(device, non_blocking=device.type == "cuda") if torch.is_tensor(v) else v)
                for k, v in value.items()
            }
        else:
            out[key] = value
    return out


@torch.inference_mode()
def denoise_batch(model, batch: dict[str, Any]) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (denoised_mean, clean) tensors with shape (B, 1, 1, H, W)."""
    batch = _move_batch_to_device(batch, model.device)

    with autocast_ctx:
        ctf_weights = (
            model.reconstructor.extractor.ctf.get_fftw_image(**batch["ctf_params"])
            .detach()
            .float()
            .abs()[:, None, ...]
        )

        model_input = batch["input"]
        if model.hparams.ctf_premultiply:
            correct_ctf = getattr(model, "_correct_ctf", None)
            if correct_ctf is None:
                raise AttributeError("Model does not implement CTF premultiply correction.")
            model_input = correct_ctf(model_input, ctf_weights)

        # if hasattr(model, "_forward_splitwise"):
        #     denoised = model._forward_splitwise(model_input)
        # else:
        denoised = model.restore_tensor(model.model(model.flatten_tensor(model_input)))

        if (
            hasattr(model.criterion, "dose_weighter")
            and model.hparams.reconstruction_mode != "denoised_full_dose"
        ):
            f_denoised = torch.fft.rfft2(denoised, norm="forward")
            f_denoised *= model.criterion.dose_weighter.weight
            f_denoised = f_denoised.mean(dim=1, keepdim=True)
            denoised_mean = torch.fft.irfft2(f_denoised, norm="forward")
        else:
            denoised_mean = denoised.mean(dim=1, keepdim=True)

    return denoised_mean, batch["clean"]


def compute_psnr(clean: torch.Tensor, denoised: torch.Tensor, model) -> tuple[float, float]:
    clean_norm = normalize_for_metrics(model._denormalize(clean.float()))
    den_norm = normalize_for_metrics(denoised.float())
    mse = ((clean_norm - den_norm) ** 2).mean().item()
    psnr = 10.0 * math.log10(1.0 / max(mse, 1e-10))
    return mse, psnr

In [ ]:
import time

results: list[dict[str, Any]] = []
start_idx = 0

if CHECKPOINT_RESULTS_CSV.exists():
    partial_df = pd.read_csv(CHECKPOINT_RESULTS_CSV)
    results = partial_df.to_dict("records")
    start_idx = len(results)
    print(f"Resuming from {start_idx} completed particles")

progress = tqdm(
    total=len(eval_loader),
    initial=start_idx,
    desc="Evaluating PSNR",
    unit="particle",
    miniters=1,
    smoothing=0.05,
)

t0 = time.perf_counter()
for batch_idx, batch in enumerate(eval_loader):
    if batch_idx < start_idx:
        continue

    denoised_mean, clean = denoise_batch(model, batch)

    for sample_idx in range(clean.shape[0]):
        sample_clean = clean[sample_idx : sample_idx + 1]
        sample_denoised = denoised_mean[sample_idx : sample_idx + 1]
        mse, psnr = compute_psnr(sample_clean, sample_denoised, model)

        particle_idx = None
        if "particle_index" in batch:
            particle_idx = int(batch["particle_index"][sample_idx])
        elif "particle_index" in eval_particles.columns:
            global_idx = batch_idx * BATCH_SIZE + sample_idx
            particle_idx = int(eval_particles.iloc[global_idx]["particle_index"])

        results.append(
            {
                "batch_idx": batch_idx,
                "sample_idx": sample_idx,
                "particle_index": particle_idx,
                "file": batch["file"][sample_idx] if "file" in batch else None,
                "mse": mse,
                "psnr": psnr,
            }
        )

    progress.update(1)

    if SAVE_EVERY_N and len(results) % SAVE_EVERY_N == 0:
        CHECKPOINT_RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(results).to_csv(CHECKPOINT_RESULTS_CSV, index=False)

progress.close()

elapsed = time.perf_counter() - t0
processed = len(results) - start_idx
if processed > 0:
    print(f"Processed {processed} particles in {elapsed / 60:.1f} min ({elapsed / processed:.2f} s/particle)")
    remaining = len(eval_loader) - len(results)
    if remaining > 0:
        print(f"Estimated remaining time: {(remaining * elapsed / processed) / 3600:.1f} h")

results_df = pd.DataFrame(results)
results_df.head()

In [ ]:
summary = {
    "n_particles": len(results_df),
    "mean_psnr": results_df["psnr"].mean(),
    "std_psnr": results_df["psnr"].std(),
    "median_psnr": results_df["psnr"].median(),
    "mean_mse": results_df["mse"].mean(),
    "checkpoint": str(CHECKPOINT_PATH),
    "config": str(CONFIG_PATH) if CONFIG_PATH else EXPERIMENT_OVERRIDE,
}

print("Summary")
for key, value in summary.items():
    print(f"  {key}: {value}")

if OUTPUT_CSV is not None:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved per-particle results to {OUTPUT_CSV.resolve()}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(results_df["psnr"], bins=40, color="#2563eb", edgecolor="white")
axes[0].set_xlabel("PSNR (dB)")
axes[0].set_ylabel("Count")
axes[0].set_title("Per-particle PSNR distribution")
axes[0].axvline(summary["mean_psnr"], color="crimson", linestyle="--", label=f"mean={summary['mean_psnr']:.2f} dB")
axes[0].legend()

axes[1].plot(results_df["psnr"].values, linewidth=0.8)
axes[1].set_xlabel("Particle index (iteration order)")
axes[1].set_ylabel("PSNR (dB)")
axes[1].set_title("PSNR per particle")

plt.tight_layout()
plt.show()